In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/PerfectPetClub.db")

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)

display(tables)

,name
0,payment
1,sqlite_sequence
2,product_category
3,product
4,subscription_type
5,subscription
6,customer
7,sale
8,sale_product
9,product_review


In [2]:
# Inspect sales, payments and reviews table structures

tables_to_check = [
    "sale",
    "sale_product",
    "payment",
    "product_review",
    "product"
]

for table in tables_to_check:
    print(f"\n========== {table.upper()} ==========")

    columns = pd.read_sql_query(
        f"PRAGMA table_info({table})",
        conn
    )

    display(columns[["name", "type"]])


========== SALE ==========


,name,type
0,id,INTEGER
1,order_date,TEXT
2,expected_date,TEXT
3,delivered_date,TEXT
4,customer_id,INTEGER
5,payment_id,INTEGER



========== SALE_PRODUCT ==========


,name,type
0,id,INTEGER
1,quantity,INTEGER
2,sale_id,INTEGER
3,product_id,INTEGER



========== PAYMENT ==========


,name,type
0,id,INTEGER
1,amount,INTEGER
2,payment_gateway,TEXT
3,transaction_reference,TEXT



========== PRODUCT_REVIEW ==========


,name,type
0,id,INTEGER
1,star_rating,INTEGER
2,product_id,INTEGER
3,customer_id,INTEGER
4,sale_id,INTEGER



========== PRODUCT ==========


,name,type
0,id,INTEGER
1,name,TEXT
2,price,INTEGER
3,product_category_id,INTEGER


In [3]:
tables_to_check = [
    "sale",
    "sale_product",
    "payment",
    "product_review",
    "product"
]

for table in tables_to_check:
    print(f"\n===== {table.upper()} =====")

    columns = pd.read_sql_query(
        f"PRAGMA table_info({table})",
        conn
    )

    print(columns[["name", "type"]].to_string(index=False))


===== SALE =====
          name    type
            id INTEGER
    order_date    TEXT
 expected_date    TEXT
delivered_date    TEXT
   customer_id INTEGER
    payment_id INTEGER

===== SALE_PRODUCT =====
      name    type
        id INTEGER
  quantity INTEGER
   sale_id INTEGER
product_id INTEGER

===== PAYMENT =====
                 name    type
                   id INTEGER
               amount INTEGER
      payment_gateway    TEXT
transaction_reference    TEXT

===== PRODUCT_REVIEW =====
       name    type
         id INTEGER
star_rating INTEGER
 product_id INTEGER
customer_id INTEGER
    sale_id INTEGER

===== PRODUCT =====
               name    type
                 id INTEGER
               name    TEXT
              price INTEGER
product_category_id INTEGER


In [4]:
# Check the sales table
print("===== SALE TABLE =====")

sale_columns = pd.read_sql_query(
    "PRAGMA table_info(sale)",
    conn
)

display(sale_columns[["name", "type"]])

# Check relationships between tables
for table in ["sale", "sale_product", "payment"]:
    print(f"\n===== {table.upper()} RELATIONSHIPS =====")

    foreign_keys = pd.read_sql_query(
        f"PRAGMA foreign_key_list({table})",
        conn
    )

    display(foreign_keys)

===== SALE TABLE =====


,name,type
0,id,INTEGER
1,order_date,TEXT
2,expected_date,TEXT
3,delivered_date,TEXT
4,customer_id,INTEGER
5,payment_id,INTEGER



===== SALE RELATIONSHIPS =====


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,payment,payment_id,id,NO ACTION,NO ACTION,NONE
1,1,0,customer,customer_id,id,NO ACTION,NO ACTION,NONE



===== SALE_PRODUCT RELATIONSHIPS =====


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,product,product_id,id,NO ACTION,NO ACTION,NONE
1,1,0,sale,sale_id,id,NO ACTION,NO ACTION,NONE



===== PAYMENT RELATIONSHIPS =====


,id,seq,table,from,to,on_update,on_delete,match


In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from extract_sales_reviews import extract_sales_and_reviews

# Run Lakshmi's extraction
sales_reviews_df = extract_sales_and_reviews(
    db_path=PROJECT_ROOT / "data" / "PerfectPetClub.db"
)

print("Total customers:", len(sales_reviews_df))
print("Dataset shape:", sales_reviews_df.shape)

display(sales_reviews_df.head())

Customers processed: 221
Sales records: 23188
Product lines: 59759
Review records: 10990
Total customers: 221
Dataset shape: (221, 9)


,customer_id,total_orders,total_payment_amount,missing_payment_count,total_items,product_list_value,review_count,avg_star_rating,one_star_reviews_count
0,1,20,74455,0,46,74455,10,2.400000,4
1,2,253,1036961,0,748,1036961,113,3.176991,19
2,3,304,1284692,0,824,1284692,134,3.082090,28
3,4,1055,9577504,0,7362,9577504,537,3.299814,83
4,5,44,179738,0,116,179738,18,3.555556,1


In [6]:
import sys
import importlib
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\DHANALAKSHI R\DataEngineeringPresentation"
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import extract_sales_reviews
importlib.reload(extract_sales_reviews)

# Run extraction, transformation and validation
sales_reviews_df = extract_sales_reviews.run_pipeline(
    db_path=PROJECT_ROOT / "data" / "PerfectPetClub.db"
)

print("Total customers:", len(sales_reviews_df))
print("Dataset shape:", sales_reviews_df.shape)

display(sales_reviews_df.head())

2026-11-11 20:41:22,094 - INFO - === Starting Sales and Reviews ETL Pipeline ===
2026-11-11 20:41:22,096 - INFO - Connecting to SQLite database: C:\Users\DHANALAKSHI R\DataEngineeringPresentation\data\PerfectPetClub.db
2026-11-11 20:41:22,653 - INFO - Customers extracted: 221
2026-11-11 20:41:22,654 - INFO - Sales extracted: 23188
2026-11-11 20:41:22,656 - INFO - Sale product records: 59759
2026-11-11 20:41:22,658 - INFO - Reviews extracted: 10990
2026-11-11 20:41:22,672 - INFO - Raw data validation completed.
2026-11-11 20:41:22,795 - INFO - Customer-level sales and review dataset created: 221 rows
2026-11-11 20:41:22,803 - INFO - Executing automated Sales and Reviews Data Quality checks...
2026-11-11 20:41:22,808 - INFO - All Sales and Reviews Data Quality checks PASSED (6/6).
2026-11-11 20:41:22,810 - INFO - === Sales and Reviews ETL Pipeline Complete ===


Total customers: 221
Dataset shape: (221, 9)


,customer_id,total_orders,total_payment_amount,missing_payment_count,total_items,product_list_value,review_count,avg_star_rating,one_star_reviews_count
0,1,20,74455,0,46,74455,10,2.400000,4
1,2,253,1036961,0,748,1036961,113,3.176991,19
2,3,304,1284692,0,824,1284692,134,3.082090,28
3,4,1055,9577504,0,7362,9577504,537,3.299814,83
4,5,44,179738,0,116,179738,18,3.555556,1


In [8]:
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\DHANALAKSHI R\DataEngineeringPresentation"
)

output_dir = PROJECT_ROOT / "output"

output_dir.mkdir(parents=True, exist_ok=True)

print("Output folder created successfully!")
print("Folder location:", output_dir)

Output folder created successfully!
Folder location: C:\Users\DHANALAKSHI R\DataEngineeringPresentation\output


In [3]:
import sys
import importlib
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\DHANALAKSHI R\DataEngineeringPresentation"
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import extract_sales_reviews
importlib.reload(extract_sales_reviews)

sales_reviews_df = extract_sales_reviews.run_pipeline(
    db_path=PROJECT_ROOT / "data" / "PerfectPetClub.db"
)

print("Total customers:", len(sales_reviews_df))
print("Dataset shape:", sales_reviews_df.shape)

display(sales_reviews_df.head())

2026-11-11 20:51:49,769 - INFO - === Starting Sales and Reviews ETL Pipeline ===
2026-11-11 20:51:49,771 - INFO - Connecting to SQLite database: C:\Users\DHANALAKSHI R\DataEngineeringPresentation\data\PerfectPetClub.db
2026-11-11 20:51:50,292 - INFO - Customers extracted: 221
2026-11-11 20:51:50,295 - INFO - Sales extracted: 23188
2026-11-11 20:51:50,297 - INFO - Sale product records: 59759
2026-11-11 20:51:50,298 - INFO - Reviews extracted: 10990
2026-11-11 20:51:50,321 - INFO - Raw data validation completed.
2026-11-11 20:51:50,472 - INFO - Customer-level sales and review dataset created: 221 rows
2026-11-11 20:51:50,476 - INFO - Executing automated Sales and Reviews Data Quality checks...
2026-11-11 20:51:50,484 - INFO - All Sales and Reviews Data Quality checks PASSED (6/6).
2026-11-11 20:51:50,486 - INFO - === Sales and Reviews ETL Pipeline Complete ===


Total customers: 221
Dataset shape: (221, 9)


,customer_id,total_orders,total_payment_amount,missing_payment_count,total_items,product_list_value,review_count,avg_star_rating,one_star_reviews_count
0,1,20,74455,0,46,74455,10,2.400000,4
1,2,253,1036961,0,748,1036961,113,3.176991,19
2,3,304,1284692,0,824,1284692,134,3.082090,28
3,4,1055,9577504,0,7362,9577504,537,3.299814,83
4,5,44,179738,0,116,179738,18,3.555556,1


In [4]:
output_dir = PROJECT_ROOT / "output"
output_dir.mkdir(parents=True, exist_ok=True)

parquet_path = output_dir / "sales_reviews_features.parquet"

sales_reviews_df.to_parquet(
    parquet_path,
    engine="pyarrow",
    index=False
)

print("Parquet saved successfully!")
print("File location:", parquet_path)

Parquet saved successfully!
File location: C:\Users\DHANALAKSHI R\DataEngineeringPresentation\output\sales_reviews_features.parquet


In [5]:
import pandas as pd

verified_df = pd.read_parquet(parquet_path)

assert verified_df.equals(sales_reviews_df)
assert verified_df["customer_id"].is_unique

print("Parquet verification PASSED!")
print("Dataset shape:", verified_df.shape)

Parquet verification PASSED!
Dataset shape: (221, 9)
